# Bank Churners — Exploratory Data Analysis

**Objective:** Understand data quality, churn rate, and candidate drivers for attrition.

**Assumptions:** Patterns are descriptive only (correlation ≠ causation). Two Naive Bayes columns are excluded as leakage per `docs/02_dataset_dictionary.md`.

### How to read this notebook
1. **Setup** — paths, constants, plotting helpers
2. **Load** — raw data snapshot
3. **Quality** — missing, Unknown, duplicates, outliers
4. **Univariate** — each feature alone
5. **Bivariate** — feature vs churn
6. **Correlation** — numeric relationships
7. **Insights** — lists handed off to modeling


In [1]:
# =============================================================================
# 0. Setup — imports, paths, and reusable plotting helpers
# =============================================================================
# EDA notebooks usually start by fixing:
#   - where data and outputs live (so paths work from repo root or notebooks/)
#   - which columns must never enter a model (leakage / IDs)
#   - a consistent visual style for all figures saved to reports/

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Reproducibility: any random sampling later will repeat the same results.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Resolve project root: Jupyter may start in /notebooks or repo root.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "BankChurners.csv").exists():
    ROOT = Path.cwd().parent
if not (ROOT / "data" / "raw" / "BankChurners.csv").exists():
    raise FileNotFoundError("Place BankChurners.csv in data/raw/")

RAW_PATH = ROOT / "data" / "raw" / "BankChurners.csv"
FIG_DIR = ROOT / "reports" / "figures"
TABLE_DIR = ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Leakage: identifiers and pre-computed model scores must not be used as features.
LEAKAGE_COLS = [
    "CLIENTNUM",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2",
]

# Feature groups drive which plots we draw (numeric vs categorical, discrete vs continuous).
NUMERIC_FEATURES = [
    "Customer_Age",
    "Dependent_count",
    "Credit_Limit",
    "Total_Revolving_Bal",
    "Avg_Open_To_Buy",
    "Months_on_book",
    "Total_Relationship_Count",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Total_Amt_Chng_Q4_Q1",
    "Total_Ct_Chng_Q4_Q1",
    "Avg_Utilization_Ratio",
]

CATEGORICAL_FEATURES = [
    "Gender",
    "Education_Level",
    "Marital_Status",
    "Income_Category",
    "Card_Category",
]

# Discrete counts are better as histograms; continuous amounts as KDE curves.
DISCRETE_NUMERIC = {
    "Dependent_count",
    "Total_Relationship_Count",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Trans_Ct",
}

# Colorblind-friendly palette: same colors in every churn vs non-churn chart.
CHURN_COLOR = "#0072B2"
NON_CHURN_COLOR = "#E69F00"
PALETTE = [NON_CHURN_COLOR, CHURN_COLOR]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
    "font.size": 11,
})

fig_counter = 0

def save_fig(name: str):
    """Save the current figure to reports/figures/ and close it (avoids memory buildup)."""
    global fig_counter
    fig_counter += 1
    path = FIG_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    return path



## 1. Data Loading

> **Learning goal:** Confirm the dataset loaded correctly (shape, columns, sample rows) before any transformation.


In [2]:
# Load the raw file once; keep df_raw unchanged for auditability.
df_raw = pd.read_csv(RAW_PATH)
print("Shape:", df_raw.shape)  # rows × columns — first sanity check
df_raw.head()  # inspect column names and example values



Shape: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,...,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,...,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,...,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,...,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,...,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,...,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


In [3]:
# Working copy: we add derived columns here without overwriting the raw snapshot.
df = df_raw.copy()

# Binary target (1 = churned) is easier for rates, correlations, and later modeling.
df["is_churn"] = (df["Attrition_Flag"] == "Attrited Customer").astype(int)
df["churn_label"] = df["is_churn"].map({0: "Existing Customer", 1: "Attrited Customer"})

churn_rate = df["is_churn"].mean()
print(f"Churn rate: {churn_rate:.2%} ({df['is_churn'].sum()} / {len(df)})")

# eda_df drops leakage columns for exploration; CLIENTNUM kept only if needed for QA.
eda_df = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns and c != "CLIENTNUM"], errors="ignore")
print("Columns excluded from EDA features:", [c for c in LEAKAGE_COLS if c in df.columns])



Churn rate: 16.07% (1627 / 10127)
Columns excluded from EDA features: ['CLIENTNUM', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']


## 2. Data Quality

> **Learning goal:** Treat data quality as a checklist—missing values, ambiguous categories, duplicate IDs, and outliers—before drawing business conclusions.


In [4]:
# --- Missing values: NaN counts per column ---
missing = df.isna().sum().sort_values(ascending=False)

# --- "Unknown" in categoricals is not NaN but still affects encoding later ---
unknown_counts = {}
for col in CATEGORICAL_FEATURES + ["Attrition_Flag"]:
    if col in df.columns and df[col].dtype == "object":
        unknown_counts[col] = int((df[col].astype(str).str.strip() == "Unknown").sum())

# Build a reusable quality table for BI / modeling handoff.
quality_rows = []
for col in df.columns:
    quality_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "missing": int(df[col].isna().sum()),
        "missing_pct": round(100 * df[col].isna().mean(), 2),
        "unknown": unknown_counts.get(col, 0),
        "n_unique": int(df[col].nunique(dropna=False)),
    })

quality_df = pd.DataFrame(quality_rows).sort_values("column")
quality_df.to_csv(TABLE_DIR / "eda_data_quality_summary.csv", index=False)
quality_df.head(10)



,column,dtype,missing,missing_pct,unknown,n_unique
1,Attrition_Flag,object,0,0.0,0,2
15,Avg_Open_To_Buy,float64,0,0.0,0,6813
20,Avg_Utilization_Ratio,float64,0,0.0,0,964
0,CLIENTNUM,int64,0,0.0,0,10127
8,Card_Category,object,0,0.0,0,4
12,Contacts_Count_12_mon,int64,0,0.0,0,7
13,Credit_Limit,float64,0,0.0,0,6205
2,Customer_Age,int64,0,0.0,0,45
4,Dependent_count,int64,0,0.0,0,6
5,Education_Level,object,0,0.0,1519,7


In [5]:
# IDs should be unique: duplicates would mean two rows for one customer.
dup_clients = df["CLIENTNUM"].duplicated().sum()
print("Duplicate CLIENTNUM:", dup_clients)

# Visualize missingness (even when counts are zero, the chart documents the check).
fig, ax = plt.subplots(figsize=(8, 4))
missing.plot(kind="barh", color=CHURN_COLOR, ax=ax)
ax.set_title("Missing values by column")
ax.set_xlabel("Count")
save_fig("missing_values_summary")



Duplicate CLIENTNUM: 0


WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/missing_values_summary.png')

**Observation:** Missing values are minimal in this snapshot; focus shifts to `Unknown` categories and class imbalance.

> **Tip:** In churn problems, class imbalance affects how you read bar charts and which metrics you will use later (e.g., recall vs accuracy).


In [6]:
# Focus only on columns that actually contain "Unknown" labels.
unknown_series = pd.Series(unknown_counts).sort_values(ascending=False)
unknown_series = unknown_series[unknown_series > 0]

fig, ax = plt.subplots(figsize=(8, 4))
unknown_series.plot(kind="bar", color=NON_CHURN_COLOR, ax=ax)
ax.set_title("Unknown category counts")
ax.set_ylabel("Rows")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
save_fig("unknown_categories_summary")



WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/unknown_categories_summary.png')

In [7]:
# Outlier rule: classic 1.5×IQR fences (exploratory; not necessarily drop rows).
outlier_summary = []
for col in NUMERIC_FEATURES:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_summary.append({"feature": col, "outliers_iqr": n_out, "pct": round(100 * n_out / len(df), 2)})

outlier_df = pd.DataFrame(outlier_summary).sort_values("outliers_iqr", ascending=False)
outlier_df.head(8)



,feature,outliers_iqr,pct
2,Credit_Limit,984,9.72
4,Avg_Open_To_Buy,963,9.51
9,Total_Trans_Amt,896,8.85
8,Contacts_Count_12_mon,629,6.21
11,Total_Amt_Chng_Q4_Q1,396,3.91
12,Total_Ct_Chng_Q4_Q1,394,3.89
5,Months_on_book,386,3.81
7,Months_Inactive_12_mon,331,3.27


## 3. Univariate Analysis

> **Learning goal:** Understand each variable *on its own* first. Univariate plots answer "what does this feature look like overall?"—not yet "how does it relate to churn?"


In [8]:
# Univariate target: always check class balance before choosing metrics.
fig, ax = plt.subplots(figsize=(7, 4))
order = ["Existing Customer", "Attrited Customer"]
counts = df["Attrition_Flag"].value_counts().reindex(order)
bars = ax.bar(counts.index, counts.values, color=[NON_CHURN_COLOR, CHURN_COLOR])
# Annotate bars so readers do not have to estimate counts from the axis alone.
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:,}", ha="center", va="bottom")
ax.set_title("Target distribution — Attrition_Flag")
ax.set_ylabel("Customers")
ax.set_xlabel("Status")
save_fig("target_distribution")



WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/target_distribution.png')

**Observation:** Roughly 16% of customers are attrited; modeling will need class-imbalance aware metrics and stratified splits.

> **Why it matters:** The target prevalence sets expectations for baseline models and sample weights.


In [9]:
# Univariate numeric: one distribution per feature (shape, skew, typical range).
for col in NUMERIC_FEATURES:
    fig, ax = plt.subplots(figsize=(7, 4))
    if col in DISCRETE_NUMERIC:
        sns.histplot(df[col], bins=range(int(df[col].min()), int(df[col].max()) + 2), color=CHURN_COLOR, ax=ax)
        ax.set_title(f"Distribution of {col} (discrete)")
    else:
        sns.kdeplot(df[col], fill=True, color=CHURN_COLOR, ax=ax)
        ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Density" if col not in DISCRETE_NUMERIC else "Count")
    save_fig(f"univariate_numeric_{col}")



In [10]:
# Univariate categorical: category mix in the full population (not yet by churn).
for col in CATEGORICAL_FEATURES:
    fig, ax = plt.subplots(figsize=(8, 4))
    order = df[col].value_counts().index
    props = df[col].value_counts(normalize=True).reindex(order)
    ax.bar(range(len(props)), props.values, color=CHURN_COLOR)
    ax.set_xticks(range(len(props)))
    ax.set_xticklabels(props.index, rotation=25, ha="right")
    ax.set_ylabel("Proportion")
    ax.set_title(f"Category mix — {col}")
    save_fig(f"univariate_categorical_{col}")



## 4. Bivariate Analysis (vs churn)

> **Learning goal:** Compare groups (churned vs existing). Here we ask: "Does this feature's distribution or category mix *differ* when the customer churned?"


In [11]:
# Bivariate numeric: compare distributions of each feature across churn groups.
# Boxplots show median, spread, and outliers — useful for behavioral variables.
bivariate_numeric = [
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Avg_Utilization_Ratio",
    "Total_Relationship_Count",
]

for col in bivariate_numeric:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(data=df, x="churn_label", y=col, hue="churn_label", palette=PALETTE, legend=False, ax=ax)
    ax.set_title(f"{col} by churn status")
    ax.set_xlabel("")
    save_fig(f"bivariate_box_{col}")



In [12]:
# Bivariate categorical: churn *rate within* each category (row-normalized crosstab).
for col in CATEGORICAL_FEATURES:
    ct = pd.crosstab(df[col], df["churn_label"], normalize="index")[["Attrited Customer"]]
    ct = ct.sort_values("Attrited Customer", ascending=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(ct.index.astype(str), ct["Attrited Customer"], color=CHURN_COLOR)
    ax.set_xlabel("Churn rate within category")
    ax.set_title(f"Churn rate by {col}")
    ax.invert_yaxis()
    save_fig(f"bivariate_churn_rate_{col}")



## 5. Correlation analysis

> **Learning goal:** Spot linear relationships among numeric features. High correlation between *inputs* informs feature selection; correlation with the *target* suggests predictive signal (not causation).


In [13]:
# Correlation matrix: linear association only — not causation.
corr_cols = NUMERIC_FEATURES + ["is_churn"]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, ax=ax)
ax.set_title("Correlation heatmap (numeric features + churn)")
save_fig("correlation_heatmap_with_target")

# Rank features by absolute correlation with churn for modeling prioritization.
top_target_corr = (
    corr["is_churn"].drop("is_churn").abs().sort_values(ascending=False).head(8)
)
print("Top |correlation| with churn:")
print(top_target_corr)



Top |correlation| with churn:
Total_Trans_Ct              0.371403
Total_Ct_Chng_Q4_Q1         0.290054
Total_Revolving_Bal         0.263053
Contacts_Count_12_mon       0.204491
Avg_Utilization_Ratio       0.178410
Total_Trans_Amt             0.168598
Months_Inactive_12_mon      0.152449
Total_Relationship_Count    0.150005
Name: is_churn, dtype: float64


**Observation:** Behavioral and change metrics tend to show stronger linear association with churn than static demographics. High correlation between credit limit and open-to-buy suggests checking multicollinearity before modeling.

> **Caution:** Correlation ≠ causation. Use these plots to generate hypotheses, not retention policies.


## 6. Key Insights

> **Learning goal:** Translate plots into actionable *lists* for the next phase (modeling): features, encoding rules, leakage exclusions, and open data issues.


In [14]:
# Consolidate handoff lists for the modeling notebook (02_modeling).
candidate_features = [c for c in eda_df.columns if c not in [
    "Attrition_Flag", "is_churn", "churn_label"
]]

# Ordinal: single numeric column with business order (Unknown often kept as 0).
ordinal_vars = {
    "Education_Level": {
        "Unknown": 0,
        "Uneducated": 1,
        "High School": 2,
        "College": 3,
        "Graduate": 4,
        "Post-Graduate": 5,
        "Doctorate": 6,
    },
    "Income_Category": {
        "Unknown": 0,
        "Less than $40K": 1,
        "$40K - $60K": 2,
        "$60K - $80K": 3,
        "$80K - $120K": 4,
        "$120K +": 5,
    },
}

# Nominal: use N-1 dummy columns to avoid the dummy variable trap.
nominal_vars = ["Gender", "Marital_Status", "Card_Category"]

print("=== Candidate features for modeling ===")
print(candidate_features)

print("\n=== Leakage / exclude ===")
print(LEAKAGE_COLS)

print("\n=== Data quality issues ===")
print("- Class imbalance (~16% churn)")
print("- Unknown categories present (see eda_data_quality_summary.csv)")
print("- IQR outliers in transaction and utilization fields (review caps/winsorization)")
print("- Potential multicollinearity among credit limit, revolving balance, and open-to-buy")

print("\n=== Feature engineering decisions ===")
print("- Drop CLIENTNUM and both Naive Bayes classifier columns")
print("- Map ordinal variables with business order starting at 1 for non-unknown levels")
print("- One-hot encode nominal variables with N-1 dummies")
print("- Keep Unknown as explicit level or impute after segment review")
print("- Stratified train/validation/test split preserving churn rate")

print("\n=== Optional hypotheses (not proven) ===")
print("- Higher inactivity and contact counts may coincide with attrition risk")
print("- Lower transaction volume and negative Q4/Q1 change may signal disengagement")



=== Candidate features for modeling ===
['CLIENTNUM', 'Customer_Age', 'Gender', 'Dependent_count', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']

=== Leakage / exclude ===
['CLIENTNUM', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']

=== Data quality issues ===
- Class imbalance (~16% churn)
- Unknown categories present (see eda_data_quality_summary.csv)
- IQR outliers in transaction and utilization fields (review caps/winsorization)
- Potential multicollinearity among credit limit, 

### Summary of main churn drivers (descriptive)

- **Behavioral signals:** Inactivity, contact frequency, and transaction amount/count differ between churned and existing customers in boxplots and category churn rates.
- **Utilization & balances:** Utilization ratio and revolving balance patterns separate groups; interpret as risk indicators, not causes.
- **Profile segments:** Education, income band, and card category show uneven churn rates across levels.

### Ordinal vs nominal encoding

| Type | Variables | Modeling note |
|------|-----------|---------------|
| Ordinal | `Education_Level`, `Income_Category` | Integer mapping with documented hierarchy; treat `Unknown` separately |
| Nominal | `Gender`, `Marital_Status`, `Card_Category` | N-1 dummy encoding |

### Limitations

- Cross-sectional data; no time-to-event modeling.
- Correlations do not justify causal retention policies.
- Leakage columns must remain excluded in `02_modeling`.


In [15]:
# Final sanity check: how many figures were exported during this run?
print(f"Figures saved: {fig_counter} files in {FIG_DIR}")
list(sorted(FIG_DIR.glob('*.png')))[:5], '...'



Figures saved: 34 files in C:\Users\Windows\Documents\Repo Folder\bank-churners-cursor-bootcamp\reports\figures


([WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/bivariate_box_Avg_Utilization_Ratio.png'),
  WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/bivariate_box_Contacts_Count_12_mon.png'),
  WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/bivariate_box_Months_Inactive_12_mon.png'),
  WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/bivariate_box_Total_Relationship_Count.png'),
  WindowsPath('C:/Users/Windows/Documents/Repo Folder/bank-churners-cursor-bootcamp/reports/figures/bivariate_box_Total_Trans_Amt.png')],
 '...')